# 05 — PatchTST + DLinear no od (EF01), regime anual (treino 2024)
PatchTST nativo (Nie et al. 2022) + controle DLinear, treino em 2024 com early stopping na val (4 fatias). Régua 02 (LSTNet anual) recarregada só para inferência. 2025 intocado.

In [1]:
import json
import random
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados" / "treino").exists())
CSV = ROOT / "dados/treino/ef01-mogi-das-cruzes_oxigenio-dissolvido_2024.csv"
OUT = ROOT / "resultados" / "05-patchtst-od"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

L, H = 8640, 288
SEASON = 288
INTERP_LIMIT = 24
VAL_SLICES = [("2024-04-19", "2024-04-28"), ("2024-07-20", "2024-07-29"),
              ("2024-09-15", "2024-09-24"), ("2024-11-20", "2024-11-24")]
LN, HN = 2016, 288
PATCH_P, PATCH_S = 48, 24
D_MODEL, NLAYERS, NHEAD, FF = 64, 3, 4, 128
BATCH, LR = 256, 1e-3
MAX_EPOCHS, PATIENCE = 60, 10
TRAIN_STRIDE, VAL_STRIDE = 8, 4
DROPOUT = 0.1
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
print("ROOT:", ROOT, "| CSV existe:", CSV.exists(), "| torch:", torch.__version__)

ROOT: /home/marcos/temporal-model | CSV existe: True | torch: 2.14.0+cpu


## 1. Carga

In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
df = df.rename(columns={"Data hora": "ds", "Oxigênio Dissolvido (mg/L)": "y"}).sort_values("ds").reset_index(drop=True)
print(df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()

(105121, 2) 2024-01-01 00:00:00 → 2024-12-31 00:00:00
faltantes: 594 (0.6%)


,ds,y
count,105121,104527.000000
mean,2024-07-01 12:00:00,4.491787
min,2024-01-01 00:00:00,0.790000
25%,2024-04-01 06:00:00,2.810000
50%,2024-07-01 12:00:00,4.870000
75%,2024-09-30 18:00:00,6.000000
max,2024-12-31 00:00:00,7.710000
std,NaN,1.725755


## 2. EDA

In [3]:
isna = df["y"].isna().to_numpy()
gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
print(f"maior gap: {gaps.max()} passos = {gaps.max()*5/60:.1f} h | gaps > 24 passos: {(gaps > 24).sum()}")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.3)
ax[0].set_title("od EF01 2024 — série completa (treino)")
ax[0].set_ylabel("od")
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("Ciclo diário")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva")

maior gap: 334 passos = 27.8 h | gaps > 24 passos: 3


fig salva


## 3. Limpeza

In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_raw = df.set_index("ds")["y"].reindex(idx)
print(f"slots na grade: {len(s_raw)} | linhas no CSV: {len(df)}")
s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")
gi = np.where(s.isna().to_numpy())[0]
blocos = np.split(gi, np.where(np.diff(gi) > 1)[0] + 1) if len(gi) else []
for g in blocos:
    print(f"  outage {s.index[g[0]]} → {s.index[g[-1]]} ({len(g)} slots)")

amostra = slice("2024-09-09", "2024-09-16")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_raw[amostra].index, s_raw[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 09–16/09")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")

slots na grade: 105121 | linhas no CSV: 105121
NaN após interpolação (limite 24): 336
  outage 2024-02-02 13:45:00 → 2024-02-02 14:45:00 (13 slots)
  outage 2024-03-11 10:55:00 → 2024-03-11 11:55:00 (13 slots)
  outage 2024-03-25 15:30:00 → 2024-03-26 17:15:00 (310 slots)
fig salva


## 4. ADF + STL (trecho limpo jul–ago)

In [5]:
trecho = s.loc["2024-07-15":"2024-08-31"].dropna()
stat, pval, *_ = adfuller(trecho.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(trecho.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")

ADF stat=-6.61 p-valor=6.3e-09 → estacionária


fig salva


## 5. Janelamento + val 4 fatias

In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy().astype(np.float32)  # float32: corta a cópia das janelas pela metade (métricas a 4 casas intactas)
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
ed = ends.date
is_val = np.zeros(len(ends), dtype=bool)
for a, b in VAL_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    m = (ed >= d0) & (ed <= d1)
    is_val |= m
    print(f"fatia {a} → {b}: {int(m.sum())} janelas válidas")
va = np.where(is_val)[0]
tr = np.where(~is_val)[0]
print(f"treino: {len(tr)} janelas | val: {len(va)} janelas | descartadas (NaN): {len(s)-L-H+1-len(X)}")
assert len(va) > 1000, "val pequena demais — reposicionar fatias!"
daily_idx = np.where((ends.time == pd.Timestamp("23:55").time()) & is_val)[0]
print("dias-âncora na val:", len(daily_idx))

fatia 2024-04-19 → 2024-04-28: 657 janelas válidas
fatia 2024-07-20 → 2024-07-29: 2880 janelas válidas
fatia 2024-09-15 → 2024-09-24: 2880 janelas válidas
fatia 2024-11-20 → 2024-11-24: 1440 janelas válidas
treino: 66073 janelas | val: 7857 janelas | descartadas (NaN): 22264
dias-âncora na val: 28


## 6. Métricas + baselines de referência

In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

Xtr, Ytr, Xva, Yva = X[tr], Y[tr], X[va], Y[va]
print("treino:", pd.DataFrame({m: metricas(Ytr, p) for m, p in cheap_preds(Xtr).items()}).T.round(4).to_string())
print("val:", pd.DataFrame({m: metricas(Yva, p) for m, p in cheap_preds(Xva).items()}).T.round(4).to_string())

treino:                       MAE    RMSE    MAPE   sMAPE
persistencia       0.4429  0.6403  8.5781  8.4512
sazonal_naive_288  0.2424  0.3741  5.1843  5.0783
media_movel_288    0.4003  0.5222  7.9572  7.8032
val:                       MAE    RMSE    MAPE   sMAPE
persistencia       0.3929  0.5728  6.4671  6.4345
sazonal_naive_288  0.1579  0.2202  2.6429  2.6531
media_movel_288    0.3260  0.4289  5.3815  5.3824


## 7. PatchTST + DLinear — treino

In [8]:
val5 = s.to_numpy().astype(np.float32)
Wln = sliding_window_view(val5, LN)
pos_end = s.index.get_indexer(ends - pd.Timedelta(minutes=5*H))
rowln = pos_end - LN + 1
print(f"janelas nativas válidas: {int((rowln >= 0).sum())}/{len(ends)} | Wln {Wln.shape}")
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
json.dump({"mode": "revin-per-window", "LN": LN, "HN": HN, "val_slices": VAL_SLICES},
          open(OUT / "modelos" / "normalizacao.json", "w"))

def monta(idxs):
    ii = np.asarray(idxs); r = rowln[ii]
    return Wln[r], Y[ii].astype(np.float32)

Xtr_, Ytr_ = monta(tr[::TRAIN_STRIDE])
Xva_, Yva_ = monta(va[::VAL_STRIDE])
print(f"treino: {Xtr_.shape} (stride {TRAIN_STRIDE}) | val: {Xva_.shape} (stride {VAL_STRIDE})")

class PatchTST(nn.Module):
    def __init__(self):
        super().__init__()
        self.N = (LN - PATCH_P) // PATCH_S + 1
        self.proj = nn.Linear(PATCH_P, D_MODEL)
        self.pos = nn.Parameter(torch.randn(1, self.N, D_MODEL) * 0.02)
        layer = nn.TransformerEncoderLayer(D_MODEL, NHEAD, FF, DROPOUT, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, NLAYERS)
        self.drop = nn.Dropout(DROPOUT)
        self.head = nn.Linear(self.N * D_MODEL, HN)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        z = self.proj(xn.unfold(1, PATCH_P, PATCH_S)) + self.pos
        z = self.enc(self.drop(z))
        y = self.head(self.drop(z.flatten(1)))
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg + mu

class DLinearLite(nn.Module):
    def __init__(self, k=25):
        super().__init__()
        self.pool = nn.AvgPool1d(k, stride=1, padding=k // 2)
        self.lin_t = nn.Linear(LN, HN)
        self.lin_s = nn.Linear(LN, HN)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        t = self.pool(xn.unsqueeze(1)).squeeze(1)
        y = self.lin_t(t) + self.lin_s(xn - t)
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg + mu

def treina(model, nome, max_ep, pat, batch):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    tr_loader = DataLoader(TensorDataset(torch.from_numpy(Xtr_), torch.from_numpy(Ytr_)),
                           batch_size=batch, shuffle=True)
    va_loader = DataLoader(TensorDataset(torch.from_numpy(Xva_), torch.from_numpy(Yva_)),
                           batch_size=512)
    print(f"{nome}: params={sum(p.numel() for p in model.parameters())}")
    best, patience, hist = float("inf"), 0, {"train": [], "val": []}
    t0 = time.time()
    for ep in range(1, max_ep + 1):
        model.train()
        tl = 0.0
        for xb, yb in tr_loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            tl += float(loss.detach()) * len(xb)
        tl /= len(tr_loader.dataset)
        model.eval()
        vl = 0.0
        with torch.no_grad():
            for xb, yb in va_loader:
                vl += float(loss_fn(model(xb), yb)) * len(xb)
        vl /= len(va_loader.dataset)
        hist["train"].append(tl); hist["val"].append(vl)
        tag = ""
        if vl < best:
            best, patience = vl, 0
            torch.save({"state": model.state_dict()}, OUT / "modelos" / nome)
            tag = " *"
        else:
            patience += 1
        print(f"ep {ep:02d} train={tl:.4f} val={vl:.4f}{tag}", flush=True)
        if patience >= pat:
            print(f"early stopping na ep {ep} (best val={best:.4f})")
            break
    print(f"{nome}: treino em {time.time()-t0:.0f}s | melhor val={best:.4f}")
    ck = torch.load(OUT / "modelos" / nome, map_location="cpu", weights_only=False)
    model.load_state_dict(ck["state"])
    model.eval()
    return model, hist

patch, hist_pt = treina(PatchTST(), "patchtst_od.pt", MAX_EPOCHS, PATIENCE, BATCH)
dlin, hist_dl = treina(DLinearLite(), "dlinear_od.pt", 30, 5, 512)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(hist_pt["train"], label="patchtst-treino")
ax.plot(hist_pt["val"], label="patchtst-val")
ax.plot(hist_dl["val"], label="dlinear-val")
ax.set_title("Loss por época (MSE nativa 5 min)")
ax.set_xlabel("época"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "figs" / "07-curvas-treino.png")
print("fig salva: 07-curvas-treino.png")

janelas nativas válidas: 73930/73930 | Wln (103106, 2016)
treino: (8260, 2016) (stride 8) | val: (1965, 2016) (stride 4)


patchtst_od.pt: params=1639010


ep 01 train=0.1991 val=0.0563 *


ep 02 train=0.1252 val=0.0444 *


ep 03 train=0.1080 val=0.0418 *


...

ep 09 train=0.0943 val=0.0387


ep 10 train=0.0935 val=0.0397


ep 11 train=0.0931 val=0.0405


ep 12 train=0.0927 val=0.0428


early stopping na ep 12 (best val=0.0384)
dlinear_od.pt: treino em 7s | melhor val=0.0384
fig salva: 07-curvas-treino.png


## 8. Régua 02 recarregada + inferência

In [9]:
class LSTNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(3, 32, kernel_size=12, stride=6)
        self.gru = nn.GRU(32, 64, batch_first=True)
        self.skipcell = nn.GRUCell(32, 32)
        self.head = nn.Linear(96, 288)
        self.ar = nn.Linear(288, 288)
        self.drop = nn.Dropout(0.1)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, xv, tod):
        mu = xv.mean(dim=1, keepdim=True); sg = xv.std(dim=1, keepdim=True).clamp_min(1e-3)
        vn = self.gamma * (xv - mu) / sg + self.beta
        f = self.drop(torch.relu(self.conv(torch.cat([vn.unsqueeze(1), tod.transpose(1, 2)], dim=1))))
        f = f.transpose(1, 2)
        _, h = self.gru(f)
        B, T, _ = f.shape
        hs = torch.zeros(B, 32, device=f.device)
        states = [hs]
        for t in range(T):
            prev = states[t - 48] if t - 48 >= 0 else states[0]
            hs = self.skipcell(f[:, t, :], prev)
            states.append(hs)
        g = self.gamma.clamp_min(1e-3)
        yn = self.head(self.drop(torch.cat([h.squeeze(0), hs], dim=1)))
        ya = self.ar(vn[:, -288:])
        return (yn + ya - self.beta) / g * sg + mu

CKPT02P = ROOT / "resultados" / "03-lstnet-od" / "modelos" / "lstnet_od.pt"
assert CKPT02P.exists(), "rode o 03-lstnet-od antes! (" + str(CKPT02P) + " ausente)"
ckpt02 = torch.load(CKPT02P, map_location="cpu", weights_only=False)
ruler = LSTNet1D().to(DEVICE)
ruler.load_state_dict(ckpt02["state"])
ruler.eval()
print("régua 02 recarregada:", CKPT02P)
SIN5 = np.sin(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
COS5 = np.cos(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
Tln = sliding_window_view(np.stack([SIN5, COS5], axis=1), LN, axis=0).transpose(0, 2, 1).astype(np.float32)

@torch.no_grad()
def prevê(model, idxs, tod=False, batch=256):
    ii = np.asarray(idxs)
    outs = []
    for b in range(0, len(ii), batch):
        xb = torch.from_numpy(Wln[rowln[ii[b:b+batch]]])
        if tod:
            tb = torch.from_numpy(Tln[rowln[ii[b:b+batch]]])
            outs.append(model(xb, tb).numpy())
        else:
            outs.append(model(xb).numpy())
    return np.concatenate(outs)

t0 = time.time()
Pt_va, Pt_d = prevê(patch, va), prevê(patch, daily_idx)
Dl_va, Dl_d = prevê(dlin, va), prevê(dlin, daily_idx)
Pn_va, Pn_d = prevê(ruler, va, tod=True), prevê(ruler, daily_idx, tod=True)
print(f"inferência em {time.time()-t0:.0f}s")
print("PatchTST val:", {k: round(v, 4) for k, v in metricas(Yva, Pt_va).items()})
print("DLinear val:", {k: round(v, 4) for k, v in metricas(Yva, Dl_va).items()})
print("LSTNet(02) val:", {k: round(v, 4) for k, v in metricas(Yva, Pn_va).items()})

régua 02 recarregada: /home/marcos/temporal-model/resultados/03-lstnet-od/modelos/lstnet_od.pt


inferência em 6s
PatchTST val: {'MAE': 0.1432, 'RMSE': 0.1909, 'MAPE': 2.3836, 'sMAPE': 2.3896}
DLinear val: {'MAE': 0.1435, 'RMSE': 0.1958, 'MAPE': 2.3812, 'sMAPE': 2.3896}
LSTNet(02) val: {'MAE': 0.138, 'RMSE': 0.1869, 'MAPE': 2.2988, 'sMAPE': 2.3065}


## 9. Tabelas

In [10]:
linhas = {m: metricas(Yva, p) for m, p in cheap_preds(Xva).items()}
linhas["lstnet(02)"] = metricas(Yva, Pn_va)
linhas["patchtst"] = metricas(Yva, Pt_va)
linhas["dlinear"] = metricas(Yva, Dl_va)
tab_va = pd.DataFrame(linhas).T.round(4)
tab_va.to_csv(OUT / "metricas_val.csv")
print("=== val ===")
print(tab_va.to_string())

Yd = Y[daily_idx]
diario = {m: metricas(Yd, cheap_preds(X[daily_idx])[m]) for m in ["persistencia", "sazonal_naive_288", "media_movel_288"]}
diario["lstnet(02)"] = metricas(Yd, Pn_d)
diario["patchtst"] = metricas(Yd, Pt_d)
diario["dlinear"] = metricas(Yd, Dl_d)
tab_d = pd.DataFrame(diario).T.round(4)
tab_d.to_csv(OUT / "metricas_val_diaria.csv")
print("=== val dias-âncora ===")
print(tab_d.to_string())

por_dia = pd.DataFrame(
    {m: [mae(Yd[k:k+1], cheap_preds(X[daily_idx])[m][k:k+1]) for k in range(len(Yd))]
     for m in ["persistencia", "sazonal_naive_288", "media_movel_288"]},
    index=[str(ends[i].date()) for i in daily_idx])
por_dia["lstnet(02)"] = [mae(Yd[k:k+1], Pn_d[k:k+1]) for k in range(len(Yd))]
por_dia["patchtst"] = [mae(Yd[k:k+1], Pt_d[k:k+1]) for k in range(len(Yd))]
por_dia["dlinear"] = [mae(Yd[k:k+1], Dl_d[k:k+1]) for k in range(len(Yd))]
por_dia.to_csv(OUT / "metricas_por_dia.csv")
print(por_dia.round(4).to_string())
print(f"\nMelhor na val: {tab_va['MAE'].idxmin()} = {tab_va['MAE'].min():.4f}")

=== val ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.3929  0.5728  6.4671  6.4345
sazonal_naive_288  0.1579  0.2202  2.6429  2.6531
media_movel_288    0.3260  0.4289  5.3815  5.3824
lstnet(02)         0.1380  0.1869  2.2988  2.3065
patchtst           0.1432  0.1909  2.3836  2.3896
dlinear            0.1435  0.1958  2.3812  2.3896
=== val dias-âncora ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.4910  0.6728  8.6095  8.0373
sazonal_naive_288  0.1579  0.2192  2.6780  2.6868
media_movel_288    0.3189  0.4243  5.3189  5.3212
lstnet(02)         0.1369  0.1935  2.3166  2.3227
patchtst           0.1594  0.2142  2.6822  2.6924
dlinear            0.1573  0.2137  2.6612  2.6646
            persistencia  sazonal_naive_288  media_movel_288  lstnet(02)  patchtst  dlinear
2024-04-26        0.2832             0.2024           0.2044      0.1657    0.1499   0.1477
2024-04-27        0.1401             0.0495           0.0772      0.0331    0.03

## 10. Figuras

In [11]:
ks = [0, len(Xtr) // 2, -1]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
cp_tr = cheap_preds(Xtr)
Pt_tr = prevê(patch, tr[ks])
Dl_tr = prevê(dlin, tr[ks])
Pn_tr = prevê(ruler, tr[ks], tod=True)
for ax, k, j in zip(axes, ks, range(3)):
    tf = pd.date_range(ends[tr[k]] - pd.Timedelta(minutes=5*(H-1)), ends[tr[k]], freq="5min")
    ax.plot(tf, Ytr[k], "k-", lw=1.5, label="real")
    ax.plot(tf, cp_tr["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, Pn_tr[j], lw=1, alpha=0.6, label="lstnet(02)")
    ax.plot(tf, Pt_tr[j], lw=1, alpha=0.9, label="patchtst")
    ax.plot(tf, Dl_tr[j], lw=1, alpha=0.7, label="dlinear")
    ax.set_title(f"origem {ends[tr[k]]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

fig, ax = plt.subplots(figsize=(8, 4))
tab_va["MAE"].sort_values().plot.barh(ax=ax)
ax.set_title("MAE na val — todos os modelos (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

fig, ax = plt.subplots(figsize=(12, 3.5))
for col, ls in [("sazonal_naive_288", "--"), ("lstnet(02)", "-."), ("patchtst", "-"), ("dlinear", ":"), ("persistencia", ":")]:
    if col in por_dia.columns:
        ax.plot(pd.to_datetime(por_dia.index), por_dia[col], ls, lw=1.1, label=col)
ax.set_title("od — MAE por dia-âncora na val")
ax.legend(fontsize=8); fig.autofmt_xdate()
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-val-dias.png")
print("figs salvas")

figs salvas


## 11. Conclusões
Checkpoints em `modelos/` para o benchmark 2025 (08).